[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/langchain-ai/langchain-academy/blob/main/module-0/basics.ipynb) [![Open in LangChain Academy](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/66e9eba12c7b7688aa3dbb5e_LCA-badge-green.svg)](https://academy.langchain.com/courses/take/intro-to-langgraph/lessons/56295530-getting-set-up-video-guide)

In [1]:
pwd

'c:\\Users\\crodr\\LanchChain_Academy\\Foundation\\Intro_LangChain\\notebooks\\module-1'

# LangChain Academy

<img src="../../pics/cover_academy.png" width=600> 



Welcome to LangChain Academy! 

https://academy.langchain.com/enrollments



## Foundation:

### Introduction to LangChain - Python

<img src="../../pics/cover_LangChain.png" width=600> 

https://academy.langchain.com/courses/take/foundation-introduction-to-langchain-python/



# Module 2: Advanced Agent

## Lesson 2: Context and State

In [1]:
pwd

'c:\\Users\\crodr\\LanchChain_Academy\\Foundation\\Intro_LangChain\\notebooks\\module-2'

In [3]:
from dotenv import load_dotenv

load_dotenv()


True

In [ ]:
from langchain.agents import AgentState


class CustomState(AgentState):
    favourite_colour: str

### Write to state

In [ ]:
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from langchain.messages import ToolMessage


@tool
def update_favourite_colour(favourite_colour: str, runtime: ToolRuntime) -> Command:
    """Update the favourite colour of the user in the state once they've revealed it."""
    return Command(
        update={
            "favourite_colour": favourite_colour,
            "messages": [
                ToolMessage(
                    "Succesfully updated favourite colour",
                    tool_call_id=runtime.tool_call_id,
                )
            ],
        }
    )

In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    "gpt-5-nano",
    tools=[update_favourite_colour],
    checkpointer=InMemorySaver(),
    state_schema=CustomState,
)

In [ ]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="My favourite colour is green")]},
    {"configurable": {"thread_id": "1"}},
)

In [9]:
from pprint import pprint

pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='My favourite colour is green', additional_kwargs={}, response_metadata={}, id='95b37a12-8014-433a-86b1-8422a7d3f897'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 283, 'prompt_tokens': 141, 'total_tokens': 424, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 256, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DjlQwJXsBQ3wb0spzQZrOcc82JGH0', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e6450-9b68-7bb3-93b7-5d40d53390dc-0', tool_calls=[{'name': 'update_favourite_colour', 'args': {'favourite_colour': 'green'}, 'id': 'call_xKa1RRsc14NZNZGmtaLMDzti', 'type': 'tool_call'}], invalid_

In [ ]:
response = agent.invoke(
    {
        "messages": [HumanMessage(content="Hello, how are you?")],
        "favourite_colour": "green",
    },
    {"configurable": {"thread_id": "10"}},
)

pprint(response)


{'favourite_colour': 'green',
 'messages': [HumanMessage(content='Hello, how are you?', additional_kwargs={}, response_metadata={}, id='181c27b4-eb5f-4085-8db6-a26650b8e184'),
              AIMessage(content='Hello! I’m doing well, thanks for asking. How about you—how are you today? If you’d like, tell me what you want to talk about or what I can help with.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 305, 'prompt_tokens': 142, 'total_tokens': 447, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 256, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DjlTovE1Nb89yh1v4EW9P57ZgjTwU', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e6453-5594-7850-ba6a-a866c0dbdceb-0', tool_calls=[], invalid

### Read state

In [ ]:
@tool
def read_favourite_colour(runtime: ToolRuntime) -> str:
    """Read the favourite colour of the user from the state."""
    try:
        return runtime.state["favourite_colour"]
    except KeyError:
        return "No favourite colour found in state"


agent = create_agent(
    "gpt-5-nano",
    tools=[update_favourite_colour, read_favourite_colour],
    checkpointer=InMemorySaver(),
    state_schema=CustomState,
)

In [ ]:
response = agent.invoke(
    {"messages": [HumanMessage(content="My favourite colour is green")]},
    {"configurable": {"thread_id": "1"}},
)

pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='My favourite colour is green', additional_kwargs={}, response_metadata={}, id='1968cacc-00ae-4fa3-9afe-7aa63ae5f486'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 347, 'prompt_tokens': 162, 'total_tokens': 509, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 320, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DjlZPG7MfmSEAKXlxfBJO7G5Pqjuy', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e6458-a385-7123-af4c-f88ee289308f-0', tool_calls=[{'name': 'update_favourite_colour', 'args': {'favourite_colour': 'green'}, 'id': 'call_VBEcmqI5pb406QOyzVPxcgd6', 'type': 'tool_call'}], invalid_

In [ ]:
response = agent.invoke(
    {"messages": [HumanMessage(content="What's my favourite colour?")]},
    {"configurable": {"thread_id": "1"}},
)

pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='My favourite colour is green', additional_kwargs={}, response_metadata={}, id='1968cacc-00ae-4fa3-9afe-7aa63ae5f486'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 347, 'prompt_tokens': 162, 'total_tokens': 509, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 320, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DjlZPG7MfmSEAKXlxfBJO7G5Pqjuy', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e6458-a385-7123-af4c-f88ee289308f-0', tool_calls=[{'name': 'update_favourite_colour', 'args': {'favourite_colour': 'green'}, 'id': 'call_VBEcmqI5pb406QOyzVPxcgd6', 'type': 'tool_call'}], invalid_